# Exercise 1 — Image Data Inspection

**CAS Deep Learning — Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Load and inspect an image dataset using PyTorch and PIL
- Understand tensor shape (C × H × W) and pixel value ranges
- Identify class imbalance in a dataset
- Spot potential data quality issues

**Fill in code where you see `# YOUR CODE HERE`.**

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    !pip install -q torchinfo gdown huggingface_hub

    ROOT_PATH = Path("/content")
    DATA_PATH = ROOT_PATH.joinpath("data")

else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

If you want to mount your Google Drive you can use the following function. This will enable you to persist data. However, it is not recommended to use this for model training because of its slow latency.

In [ ]:
def mount_drive(
    drive_mount_path: str = "/content/drive", data_path: str = "MyDrive/cas-dl-compvis"
) -> tuple[Path, Path]:
    from google.colab import drive

    root_path = Path(drive_mount_path)
    drive.mount(str(root_path))
    data_path = root_path.joinpath(data_path)
    return root_path, data_path


# GDRIVE_ROOT_PATH, GDRIVE_DATA_PATH = mount_drive()

## Imports

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import make_grid

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Part 1 — Load a Single Image

We start by loading a camera trap dataset and looking at a single image.

In [ ]:
def ensure_dataset(
    data_path,
    archive_name,
    extracted_subdir,
    *,
    gdrive_id=None,
    hf_repo=None,
):
    """Check-then-download a dataset archive. Returns the extracted directory.

    Tries Google Drive first (gdown), falls back to Hugging Face Hub. Supports
    .tar.gz and .zip. Idempotent — skips if the extracted directory exists.
    """
    data_path = Path(data_path)
    dataset_dir = data_path / extracted_subdir
    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    data_path.mkdir(parents=True, exist_ok=True)
    archive_path = data_path / archive_name

    if not archive_path.exists():
        errors = []
        if gdrive_id is not None:
            try:
                import gdown

                url = f"https://drive.google.com/uc?id={gdrive_id}"
                print(f"Downloading {archive_name} from Google Drive ...")
                gdown.download(url, str(archive_path), quiet=False, fuzzy=True)
                if not archive_path.exists():
                    raise RuntimeError("gdown reported success but file is missing")
            except Exception as exc:
                errors.append(f"gdown: {exc}")
                archive_path.unlink(missing_ok=True)

        if not archive_path.exists() and hf_repo is not None:
            try:
                from huggingface_hub import hf_hub_download

                print(f"Downloading {archive_name} from Hugging Face ({hf_repo}) ...")
                cached = hf_hub_download(hf_repo, archive_name, repo_type="dataset")
                if Path(cached).resolve() != archive_path.resolve():
                    archive_path.write_bytes(Path(cached).read_bytes())
            except Exception as exc:
                errors.append(f"hf_hub_download: {exc}")

        if not archive_path.exists():
            sources = ", ".join(errors) if errors else "no sources configured"
            raise RuntimeError(f"Could not fetch {archive_name}. Tried: {sources}.")

    print(f"Extracting {archive_path.name} to {data_path} ...")
    if archive_name.endswith((".tar.gz", ".tgz")):
        import tarfile

        with tarfile.open(archive_path) as tar:
            tar.extractall(data_path)
    elif archive_name.endswith(".zip"):
        import zipfile

        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(data_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_name}")

    assert dataset_dir.exists(), (
        f"Extraction of {archive_path.name} did not produce {dataset_dir}."
    )
    print(f"Ready: {dataset_dir}")
    return dataset_dir


# Load the Serengeti balanced dataset. Downloads from Google Drive (primary)
# or Hugging Face Hub (fallback) if not already on disk.
DATASET_PATH = ensure_dataset(
    DATA_PATH,
    archive_name="ser_balanced.tar.gz",
    extracted_subdir="ser_balanced",
    gdrive_id="1iRlZue4-Udg_lA9RCYtg3zhctqbbPhX7",
    hf_repo="marco-willi/ser_balanced",
)

train_data = ImageFolder(root=DATASET_PATH / "train")

print(f"Train size: {len(train_data)}")
print(f"Classes:    {train_data.classes}")

Load the first image from the dataset. Print its type, size, mode, and label. Then display it with `plt.imshow`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


Convert the image to a NumPy array and print its shape, dtype, and value range.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


**Question:** `transforms.ToTensor()` converts a PIL image to a PyTorch tensor. What shape and value range do you expect?

<details>
<summary>Click to reveal answer</summary>

- **Shape:** `(C, H, W)` — PyTorch uses **channels-first** ordering, so a 768×1024 RGB image becomes `(3, 768, 1024)`.
- **Values:** Scaled from `[0, 255]` (uint8) to `[0.0, 1.0]` (float32).

This is different from NumPy/PIL which use `(H, W, C)` — channels last.

</details>

Convert the image to a PyTorch tensor using `transforms.ToTensor()` and verify the shape and value range.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert img_tensor.ndim == 3, f"Expected 3D tensor, got {img_tensor.ndim}D"
assert img_tensor.shape[0] == 3, f"Expected 3 channels, got {img_tensor.shape[0]}"
assert img_tensor.min() >= 0 and img_tensor.max() <= 1, "Values should be in [0, 1]"
print("OK")

## Part 2 — Dataset Exploration

Let's explore the class distribution of the dataset. How balanced is it?

Count the number of images per class using `train_data.classes` and `train_data.targets`. Print each class name and its count.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert len(class_names) > 0, "No classes found"
assert sum(label_counts.values()) == len(train_data), "Counts don't sum to dataset size"
print("OK")

Visualize the class distribution as a bar chart.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


**Question:** Is the dataset class-balanced? What could go wrong during training?

<details>
<summary>Click to reveal answer</summary>

The "empty" class has far fewer samples than the animal classes — the dataset is **imbalanced** (4:1 ratio). This can cause:
1. **Bias towards majority classes** — the model may learn to always predict the common classes
2. **Poor recall on minority classes** — rare classes are underrepresented in training
3. **Misleading accuracy** — high overall accuracy can hide poor per-class performance

Solutions: class weighting, oversampling, undersampling, or stratified splits.

</details>

## Part 3 — Transforms and Normalization

Deep learning pipelines resize images to a fixed size and normalize pixel values. We use ImageNet statistics as the standard.

Define two transform pipelines:
- **`train_transforms`**: `Resize(256)` → `RandomCrop(224)` → `RandomHorizontalFlip` → `ToTensor` → `Normalize` (ImageNet stats)
- **`val_transforms`**: `Resize(256)` → `CenterCrop(224)` → `ToTensor` → `Normalize` (ImageNet stats)

ImageNet statistics: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
img_transformed = train_transforms(image)
assert img_transformed.shape == (3, 224, 224), (
    f"Expected (3, 224, 224), got {img_transformed.shape}"
)
print("OK")

**Question:** What do you think the image will look like after ImageNet normalization? Will it still look like a normal photo?

In [ ]:
# Let's compare: original vs resized vs normalized
simple_transforms = transforms.Compose(
    [transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()]
)
img_simple = simple_transforms(image)
img_normalized = val_transforms(image)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

_ = axes[0].imshow(image)
_ = axes[0].set_title("Original")
_ = axes[0].axis("off")

_ = axes[1].imshow(img_simple.permute(1, 2, 0).numpy())
_ = axes[1].set_title("Resized + CenterCrop\n(values in [0, 1])")
_ = axes[1].axis("off")

# Show normalized image directly — it looks wrong on purpose!
img_norm_display = img_normalized.permute(1, 2, 0).numpy()
_ = axes[2].imshow(np.clip(img_norm_display, 0, 1))
_ = axes[2].set_title("After ImageNet normalization\n(looks wrong!)")
_ = axes[2].axis("off")

_ = plt.tight_layout()
plt.show()

print(
    f"Normalized tensor — min: {img_normalized.min():.2f}, max: {img_normalized.max():.2f}"
)

<details>
<summary>Why does the normalized image look wrong?</summary>

After normalization, pixel values are **no longer in [0, 1]** — they are centered around 0 with values roughly in [-2, +3]. Matplotlib expects values in [0, 1] for floats, so the image looks washed out or clipped.

**This is expected and correct for model training.** The normalization ensures that each channel has roughly zero mean and unit variance (based on ImageNet statistics), which helps gradient-based optimization converge faster.

To display a normalized image correctly, you must **denormalize** it first (reverse the mean/std scaling).

</details>

## Part 4 — Inspect a Batch

We create a `DataLoader` to load batches of transformed images — the format models actually consume.

Create an `ImageFolder` dataset with `val_transforms` applied, then wrap it in a `DataLoader` with `batch_size=8`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


**Question:** What shape do you expect for a batch of 8 images?

<details>
<summary>Click to reveal answer</summary>

`(8, 3, 224, 224)` — the batch dimension is added as the first axis: **(B, C, H, W)**.

</details>

Load one batch from `train_loader` and print its shape, dtype, and value range.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert images_batch.shape == (batch_size, 3, 224, 224), f"Got {images_batch.shape}"
assert labels_batch.shape == (batch_size,), f"Got {labels_batch.shape}"
print("OK")

In [ ]:
# Denormalize for display
def denormalize(batch):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return torch.clamp(batch * std + mean, 0, 1)


grid = make_grid(denormalize(images_batch), nrow=4, pad_value=1)
fig, ax = plt.subplots(figsize=(12, 6))
_ = ax.imshow(grid.permute(1, 2, 0).numpy())
_ = ax.set_title(f"Batch of {batch_size} training images (denormalized for display)")
_ = ax.axis("off")
_ = plt.tight_layout()
plt.show()

## Summary

In this exercise you learned to:
1. **Load images** with PIL and convert to PyTorch tensors (C, H, W)
2. **Explore a dataset** — check class distributions and spot imbalance
3. **Apply transforms** — resize, augment, and normalize with ImageNet statistics
4. **Create DataLoaders** — load batches of shape (B, C, H, W) for training

**Key takeaways:**
- PyTorch uses channels-first: `(C, H, W)`, not `(H, W, C)`
- Normalized images look wrong when displayed — you must denormalize first
- Class imbalance is common in real datasets and requires special handling
